# Forest Plot Extraction — Eval

Reads `table_catalog.csv`, filters for `table_type=forest_plot` and `status=complete`, runs extraction on each entry, compares to ground truth, and prints accuracy (cell_accuracy + row_match).

In [34]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
from shared.client import client, DEFAULT_MODEL
from shared.pdf import render_pages
from shared.eval import compare_tables, print_accuracy_summary
from agents.forest_plot.extract import extract_forest_plots_from_page, stitch_forest_plot_results
from agents.router.classify import classify_page

In [35]:
MODEL = DEFAULT_MODEL

catalog = pd.read_csv("../table_catalog.csv")
catalog["status"] = catalog["status"].str.strip()
forest_entries = catalog[catalog["table_type"] == "forest_plot"].copy()
# Only eval entries with status=not started
eval_entries = forest_entries[forest_entries["status"] == "not started"]
print(f"Forest plot entries to evaluate: {len(eval_entries)}")
eval_entries

Forest plot entries to evaluate: 1


,paper_id,page,figure_id,table_type,variation,description,ground_truth_path,cell_accuracy,row_match,notes,status
3,diabetes_obesity_metabolism_2023,6,fig3b,forest_plot,study_level_simple,Gla-300 vs Gla-100 risk ratio for hypoglycemia...,papers/Diabetes Obesity Metabolism - 2023 - Jo...,NaN,NaN,NaN,not started


In [36]:
results = []
os.makedirs("../output", exist_ok=True)

for _, row in eval_entries.iterrows():
    paper_id = row["paper_id"]
    page = int(row["page"])
    figure_id = row["figure_id"]
    gt_path = os.path.join("..", row["ground_truth_path"])

    # Derive plot index from figure_id suffix: fig3a→0, fig3b→1, fig3c→2, etc.
    suffix = figure_id[-1] if figure_id[-1].isalpha() else "a"
    plot_index = ord(suffix) - ord("a")

    # Derive PDF path from ground_truth_path: papers/<folder>/ground_truth/... → papers/<folder>/paper.pdf
    paper_dir = os.path.dirname(os.path.dirname(gt_path))
    pdf_path = os.path.join(paper_dir, "paper.pdf")

    print(f"\n--- {paper_id} p{page} {figure_id} ({row['description']}) ---")

    # Render page
    images = render_pages(pdf_path, [page])
    page_image = images[page]

    # Step 1: Router classifies the page
    classification = classify_page(client, MODEL, page_image)
    forest_plot_entries = [t for t in classification.tables if t.type == "forest_plot"]
    print(f"  Router found {len(classification.tables)} table(s), {len(forest_plot_entries)} forest plot(s)")
    for entry in forest_plot_entries:
        print(f"    - {entry.label}: {entry.description}")

    if not forest_plot_entries:
        print("  No forest plots classified on this page!")
        continue

    # Use the matching router entry's instruction for the target figure
    router_index = min(plot_index, len(forest_plot_entries) - 1)
    instruction = forest_plot_entries[router_index].instruction
    print(f"  Using router entry {router_index} for {figure_id}: {forest_plot_entries[router_index].label}")

    # Step 2: Extract using instruction from router
    page_result = extract_forest_plots_from_page(client, MODEL, page_image, instruction=instruction)
    page_results = {page: page_result}
    dfs = stitch_forest_plot_results(page_results)

    if not dfs:
        print("  No forest plots extracted!")
        continue

    # Save all extracted plots to CSV
    for i, (label, df_ext, ftr) in enumerate(dfs):
        out_path = f"../output/{paper_id}_p{page}_{figure_id}_plot{i}.csv"
        df_ext.to_csv(out_path, index=False)
        print(f"  Saved: {out_path}")

    # Compare first extracted plot to ground truth (instruction targets the specific figure)
    _, df_extracted, footer = dfs[0]
    print(f"  Comparing extracted plot 0 for {figure_id}")
    if footer:
        print(f"  Footer: {footer}")
    df_truth = pd.read_csv(gt_path)
    result = compare_tables(df_truth, df_extracted)
    result["paper_id"] = paper_id
    result["page"] = page
    result["figure_id"] = figure_id
    results.append(result)
    print_accuracy_summary(result)


--- diabetes_obesity_metabolism_2023 p6 fig3b (Gla-300 vs Gla-100 risk ratio for hypoglycemia (events/total)) ---
  Router found 4 table(s), 4 forest plot(s)
    - Figure 1 (A): A forest plot comparing the mean difference between Gla-300 and Gla-100 groups across various clinical studies.
    - Figure 1 (B): A forest plot showing mean difference comparisons between Gla-300 and Gla-100 for a second set of data points.
    - Figure 1 (C): A forest plot presenting the mean difference results for Gla-300 versus Gla-100 across seven studies.
    - Figure 1 (D): A forest plot showing the mean difference between Gla-300 and Gla-100 for six clinical trials.
  Using router entry 1 for fig3b: Figure 1 (B)
  Saved: ../output/diabetes_obesity_metabolism_2023_p6_fig3b_plot0.csv
  Comparing extracted plot 0 for fig3b
  Footer: ['Heterogeneity: Tau² = 0.00; Chi² = 7.09, df = 9 (P = .63); I² = 0%', 'Test for overall effect: Z = 11.19 (P < .00001)', '-1 -0.5 0 0.5 1', 'Favours Gla-300 Favours Gla-100'

,Row,Column,Ground Truth,Extracted
0,0,(b),study or subgroup,
1,0,unnamed: 7,weight,



PER-COLUMN ACCURACY


,Column,Total Cells,Correct,Errors,Accuracy
0,(b),12,11,1,91.7%
1,gla-300,12,12,0,100.0%
2,unnamed: 2,12,12,0,100.0%
3,unnamed: 3,12,12,0,100.0%
4,gla-100,12,12,0,100.0%
5,unnamed: 5,12,12,0,100.0%
6,unnamed: 6,12,12,0,100.0%
7,unnamed: 7,12,11,1,91.7%
8,mean difference,12,12,0,100.0%


In [37]:
# Aggregate accuracy
if results:
    total_cells = sum(r["total_cells"] for r in results)
    correct_cells = sum(r["correct_cells"] for r in results)
    total_rows = sum(r["rows_compared"] for r in results)
    rows_correct = sum(r["rows_fully_correct"] for r in results)
    print(f"\n{'=' * 60}")
    print(f"AGGREGATE CELL ACCURACY: {correct_cells}/{total_cells} = {correct_cells/total_cells:.1%}")
    print(f"AGGREGATE ROW MATCH:     {rows_correct}/{total_rows} = {rows_correct/total_rows:.1%}")
    print(f"{'=' * 60}")
else:
    print("No entries evaluated.")


AGGREGATE CELL ACCURACY: 106/108 = 98.1%
AGGREGATE ROW MATCH:     11/12 = 91.7%
